In [ ]:
# ### Cell 1: Imports and Environment Setup ###

import os
from dotenv import load_dotenv

# Import LangChain components
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_community.llms import HuggingFaceHub
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

load_dotenv()
hf_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

# Check if the API key is available
if not hf_api_token:
    raise ValueError("Hugging Face API token not found. Please set it in your .env file.")
else:
    print("Libraries imported and Hugging Face API token loaded successfully.")

Libraries imported and Hugging Face API token loaded successfully.


In [2]:
# ### Cell 2: Constants and Custom Prompt Template ###

# Define paths for documents and the persistent vector store
PDF_DOCS_PATH = "./documents"
CHROMA_PERSIST_PATH = "./db2/chroma_store"

# This template guides the LLM's behavior.
custom_prompt_template = """
You are a knowledgeable and trustworthy medical information expert.
Your role is to synthesize the information from the provided medical text to answer the user's question directly and authoritatively.
Act as if you know the information yourself, but you MUST base your entire answer strictly on the text provided.
Do not mention that you are getting the information from a provided text.
If the answer is not contained within the text, you must state: "I do not have information on that specific topic."

Provided Medical Text: {context}
Chat History: {chat_history}

User's Question: {question}
Expert Answer:
"""

def create_custom_prompt():
    """Creates a custom prompt template for the RAG chain."""
    return PromptTemplate(
        template=custom_prompt_template,
        input_variables=["context", "chat_history", "question"]
    )

print("Constants and prompt template function defined.")

Constants and prompt template function defined.


In [ ]:
# # ### Cell 3: Vector Store Function (with Hugging Face Embeddings) ###

# def get_vectorstore():
#     """
#     Processes PDF documents using a Hugging Face model for embeddings.
#     Creates or loads a persistent Chroma vector store.
#     """
#     # Use a Hugging Face model for creating embeddings
#     # sentence-transformers/all-MiniLM-L6-v2 is a popular and efficient choice
#     embeddings = HuggingFaceInferenceAPIEmbeddings(
#         model_name="sentence-transformers/all-MiniLM-L6-v2",
#         api_key=hf_api_token 
#     )

#     if os.path.exists(CHROMA_PERSIST_PATH):
#         # Load the existing vector store
#         print("Loading existing vector store...")
#         vector_store = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)
#     else:
#         # Create a new vector store
#         print("Creating new vector store using Hugging Face...")
        
#         pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
#         if not pdf_files:
#             raise FileNotFoundError(f"No PDF files found in '{PDF_DOCS_PATH}'.")

#         documents = []
#         for pdf_file in pdf_files:
#             loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
#             documents.extend(loader.load())
        
#         text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
#         chunks = text_splitter.split_documents(documents)
        
#         # This will now make API calls to Hugging Face for each chunk.
#         # For a large number of chunks, you might need a delay here.
#         vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PERSIST_PATH)
#         print("Vector store created successfully.")
    
#     return vector_store

# print("Vector store function with Hugging Face embeddings defined.")

Vector store function with Hugging Face embeddings defined.


In [ ]:
import time
def get_vectorstore():
    """
    Create or load a Chroma vector store with Hugging Face embeddings.
    Handles both fresh creation and incremental updates with throttling.
    """
    embeddings = HuggingFaceInferenceAPIEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        api_key=hf_api_token
    )

    if os.path.exists(CHROMA_PERSIST_PATH):
        print("Loading existing vector store...")
        vector_store = Chroma(
            persist_directory=CHROMA_PERSIST_PATH,
            embedding_function=embeddings,
            collection_name="medical_docs"
        )
    else:
        print("Creating new vector store with throttling...")

        pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
        if not pdf_files:
            raise FileNotFoundError(f"No PDF files found in '{PDF_DOCS_PATH}'.")

        documents = []
        for pdf_file in pdf_files:
            loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
            documents.extend(loader.load())
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = text_splitter.split_documents(documents)
        total_chunks = len(chunks)
        print(f" Total chunks to be processed: {total_chunks}")

        # Initialize with first chunk
        vector_store = Chroma.from_documents(
            documents=[chunks[0]],
            embedding=embeddings,
            persist_directory=CHROMA_PERSIST_PATH,
            collection_name="medical_docs"
        )

        # Add the rest one by one
        for i, chunk in enumerate(chunks[1:], start=2):
            vector_store.add_documents([chunk])
            print(f"  > Embedded chunk {i} of {total_chunks}")
            time.sleep(1)

        print(" Vector store created successfully with throttling!")

    return vector_store



In [4]:
# ### Cell 4: Conversational Chain Function (with Hugging Face) ###

def get_conversational_rag_chain(vector_store):
    """
    Creates the main conversational retrieval chain using Hugging Face for generation.
    """
    # Use the Hugging Face Hub for generating answers
    llm = HuggingFaceHub(
        repo_id="EleutherAI/gpt-neox-20b", 
        model_kwargs={"temperature": 0.5, "max_new_tokens": 250},
        huggingfacehub_api_token=hf_api_token
    )

    # Set up memory to store chat history
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Create the retriever from the vector store
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # Create the custom prompt
    prompt = create_custom_prompt()

    # Create the conversational chain
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": prompt},
        verbose=False
    )
    
    return chain

print("Conversational RAG chain function with Hugging Face model defined.")

Conversational RAG chain function with Hugging Face model defined.


In [5]:
# ### Cell 5: Initialization ###

# Note: Make sure the Ollama application is running on your computer.
vector_store = get_vectorstore()
qa_chain = get_conversational_rag_chain(vector_store)


C:\Users\akmal\AppData\Local\Temp\ipykernel_33964\1512492576.py:7: LangChainDeprecationWarning: The class `HuggingFaceInferenceAPIEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpointEmbeddings``.
  embeddings = HuggingFaceInferenceAPIEmbeddings(


🆕 Creating new vector store with throttling...
📑 Total chunks to be processed: 371


KeyError: 0

In [6]:
# ### Cell 6: Interactive Chat ###

# Initialize chat history once.
try:
    chat_history
except NameError:
    chat_history = []

# --- ASK YOUR QUESTION HERE ---
question = "What are the early symptoms of Type 2 Diabetes?"
# --------------------------

if question.lower() == 'exit':
    print("Chat ended.")
else:
    # Get the result from the chain
    result = qa_chain({"question": question, "chat_history": chat_history})
    
    # Append the current question and the answer to the history
    chat_history.append((question, result["answer"]))
    
    # Print the answer
    print("AI:", result["answer"])

C:\Users\akmal\AppData\Local\Temp\ipykernel_17952\2015079912.py:17: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"question": question, "chat_history": chat_history})


KeyError: 0